# JERN — Just An Experimental Response Network

A small GPT-style transformer trained from scratch on character-level conversation data, plus an instruct LLM fine-tuned on the same conversations for the assistant server.

This notebook runs the whole pipeline on Google Colab (use **Runtime → Change runtime type → GPU**):

1. Get the code (clone the repo or upload it)
2. Prepare the character-level dataset
3. Train the from-scratch JERN model
4. Sample from it / chat with it
5. (Optional) Fine-tune the instruct chat model used by the server

Everything goes through `main.py`, exactly as it does locally.

## 1. Setup

In [ ]:
!nvidia-smi -L || echo 'No GPU: training will be slow on CPU'

In [ ]:
import os

REPO_URL = 'https://github.com/<your-user>/<your-repo>.git'  # <- set this, or leave as-is and upload a zip below
REPO_DIR = '/content/jern'

if not os.path.exists(REPO_DIR):
    if '<your-user>' in REPO_URL:
        from google.colab import files
        print('Upload a zip of the project (the folder containing main.py)')
        uploaded = files.upload()
        os.makedirs(REPO_DIR, exist_ok=True)
        !unzip -q -o "{list(uploaded)[0]}" -d "{REPO_DIR}"
        # handle zips that wrap everything in a single top-level folder
        entries = os.listdir(REPO_DIR)
        if 'main.py' not in entries and len(entries) == 1:
            REPO_DIR = os.path.join(REPO_DIR, entries[0])
    else:
        !git clone -q "{REPO_URL}" "{REPO_DIR}"

%cd {REPO_DIR}
!ls

In [ ]:
%pip install -q -r requirements.txt

## 2. Prepare the data

The training data lives in `data/chat/conversations.txt` as `User:` / `JERN:` turns separated by `---`. Edit it to teach JERN new things, then re-run this cell.

In [ ]:
!head -n 12 data/chat/conversations.txt

In [ ]:
!python main.py prepare

## 3. Train JERN from scratch

`model/jern/network.py` defines the transformer; `model/jern/train.py` the training loop. Any field of `TrainConfig` can be overridden with `--key=value`.

In [ ]:
!python main.py train --out_dir=out-jern-char --max_iters=2000 --lr_decay_iters=2000 \
    --n_layer=4 --n_head=4 --n_embd=128 --block_size=128 --batch_size=16 --dropout=0.0

## 4. Sample and chat

In [ ]:
!python main.py sample --out_dir=out-jern-char --start="User: hello" --num_samples=3 --max_new_tokens=200

In [ ]:
# Programmatic chat with the char-level model (the CLI version is `python main.py chat`).
import sys
sys.path.insert(0, '.')
from model.jern.sample import Chat, SampleConfig, Sampler

config = SampleConfig(out_dir='out-jern-char', max_new_tokens=200, temperature=0.3)
sampler = Sampler(config)
jern = Chat(sampler.model, sampler.tokenizer, config)

for message in ['hello', 'what is your name', 'who made you', 'bye']:
    print(f'User: {message}')
    print(f'JERN: {jern.reply(message)}')

## 5. Fine-tune the instruct chat model (optional)

The assistant server (`python main.py serve`) uses `JERNChat`: a small instruct LLM (`SmolLM2-360M-Instruct` by default) fine-tuned on the same conversation file so it answers as JERN. Fine-tuning on Colab is much faster than on a laptop. Download the resulting `out-jern.zip`, extract it to `export/out-jern/` next to `main.py`, and run `python main.py serve --out_dir=export/out-jern`.

Tip: with the `colab` CLI installed locally, `python main.py colab --gpu=T4 -- finetune` does all of this from your terminal.

In [ ]:
import sys
sys.path.insert(0, '.')
from app.server.assistant import AppConfig
from model.jern.chat import JERNChat

config = AppConfig(train=True, out_dir='out-jern', epochs=2, batch_size=2)
jern = JERNChat(config).load(system_prompt=config.system_prompt)

print(jern.generate([
    {'role': 'system', 'content': config.system_prompt},
    {'role': 'user', 'content': 'who are you?'},
]))

In [ ]:
!zip -q -r out-jern.zip out-jern
from google.colab import files
files.download('out-jern.zip')